In [1]:
import sys
from pathlib import Path
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\examples')
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as ag
assert ag.__version__ == '2.1.3'
assert 'site-packages' in Path(ag.__file__).parts


# System: coordinar Agents

Los mismos tres flujos, ahora con Agents como pasos.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as ag


In [3]:
@ag.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@ag.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = ag.skill(name="arithmetic", tools=[increment, double])
increase = ag.Action("increase", "increment")
multiply = ag.Action("multiply", "double")

def choose_next(data):
    return "multiply" if data["value"] >= 0 else "END"

def combine(branches):
    return {"value": sum(row["value"] for row in branches.values())}

In [4]:
for backend in ("native", "langgraph"):
    incrementer = ag.Agent("incrementer", increase, skills=(arithmetic,), backend=backend)
    doubler = ag.Agent("doubler", multiply, skills=(arithmetic,), backend=backend)
    sequential = ag.System("sequential", ag.Sequence((incrementer, doubler)), backend=backend)
    parallel = ag.System("parallel", ag.Parallel((incrementer, doubler), join=combine), backend=backend)
    decision = ag.System("decision", ag.Graph(
        start="increase", nodes=(("increase", incrementer), ("multiply", doubler)),
        edges=(("increase", choose_next), ("multiply", "END")),
    ), backend=backend)
    for system, expected in ((sequential, 8), (parallel, 10), (decision, 8)):
        result = system.run({"value": 3})
        assert result.ok and result.data == {"value": expected}
        print(backend, system.name, result.data)
    assert decision.run({"value": -3}).data == {"value": -2}

native sequential {'value': 8}
native parallel {'value': 10}
native decision {'value': 8}


langgraph sequential {'value': 8}
langgraph parallel {'value': 10}
langgraph decision {'value': 8}


In [5]:
assert blocked_attempts() == 0